# pyALDVC tutorial: a complete run on a volume pair

This notebook walks through one AL-DVC analysis: loading two volumes, choosing
parameters, running the pipeline with a checkpoint directory, reading the result
(status codes, ZNCC, uncertainty), plotting displacement and strain slices and
exporting to ParaView. Set `REFERENCE` and `DEFORMED` to your own files (TIFF
stack, slice folder, MATLAB `.mat`, `.npy`); leave them `None` to run on a
synthetic speckle pair with a known deformation.

The same steps are in `examples/scripting/tutorial_real_data.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from al_dvc import dvcpara_default, load_volume, run_aldvc
from al_dvc.core.data_structures import STATUS_NAMES
from al_dvc.export import export_npz, export_report, export_vtk
from al_dvc.io.volume_ops import memory_model
from al_dvc.viz.slices import plot_field_slices, plot_volume_slices

REFERENCE = None  # e.g. "scan/ref.tif"
DEFORMED = None  # e.g. "scan/deformed.tif"
OUTPUT = "tutorial_output"

## 1. Volumes

`load_volume` returns an `(nz, ny, nx)` array of the file's dtype. MATLAB
`vol(x, y, z)` arrays are transposed automatically. Without files, a
synthetic pair with a 2 % affine deformation is generated so that the result
can be checked against the truth.

In [ ]:
if REFERENCE and DEFORMED:
    ref, dfm, truth = load_volume(REFERENCE), load_volume(DEFORMED), None
else:
    from al_dvc.synthetic import affine_displacement, generate_speckle_volume, warp_volume_lagrangian

    shape = (96, 104, 112)  # (nz, ny, nx)
    centre = tuple((s - 1) / 2 for s in shape[::-1])  # (x, y, z)
    F = np.array([[0.02, 0.004, 0.0], [0.003, -0.01, 0.002], [0.0, -0.002, 0.01]])  # displacement gradient
    truth = affine_displacement(F, (1.3, -0.7, 0.4), centre)
    ref = generate_speckle_volume(shape, sigma=2.0, seed=11)
    dfm = warp_volume_lagrangian(ref, truth)
print(ref.shape, ref.dtype, dfm.shape, dfm.dtype)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_volume_slices(ref, fig=fig, axes=axes, title="reference")

## 2. Parameters

`dvcpara_default` returns a validated, immutable parameter set. The subset
size (`winsize`) should span a few speckles; the node spacing (`winstepsize`)
is typically half of it. `voxel_size` and `units` scale the exported fields.
`memory_model` tells how much memory the pair will need
(`gradient_mode="on_the_fly"` cuts it from 21 to 9 bytes per voxel).

In [ ]:
para = dvcpara_default(
    winsize=24,
    winstepsize=12,
    voxel_size=(1.0, 1.0, 1.0),
    units="voxel",
    search_radius=8,  # NCC search half-width at the coarsest pyramid level
    interp_method="cubic",  # "bspline" is more accurate, "linear" fastest
    use_global_step=True,  # AL-DVC: subsets + global compatibility (ADMM)
    strain_method="plane_fit",
)
mem = memory_model(ref.shape, para.gradient_mode, para.interp_method)
print(f"{mem['bytes_per_voxel']:.0f} bytes/voxel -> {mem['total_gb']:.2f} GB resident for this pair")

## 3. Run

`progress_fn` receives `(fraction, message)`; `checkpoint_dir` makes an
interrupted multi-frame run resumable (finished frames are reused on the next
call with the same directory).

In [ ]:
result = run_aldvc(
    para,
    [ref, dfm],
    progress_fn=lambda frac, msg: print(f"[{100 * frac:5.1f}%] {msg}"),
    checkpoint_dir=f"{OUTPUT}/checkpoints",
)
print({k: round(v, 2) for k, v in result.timings.items() if not k.startswith("frame_")})

## 4. What came out

`result.dvc_mesh` holds the node grid (`coordinates` (N, 3) = [x, y, z] in
voxels, `grid_shape` (nz, ny, nx), `to_grid(values)`). `result.result_disp[k]`
is the k-th frame pair: `U` (N, 3) in voxels, `F` (N, 3, 3), `status` codes,
`zncc`, the per-node uncertainty `U_std` and the ADMM diagnostics.

In [ ]:
mesh = result.dvc_mesh
fr = result.result_disp[0]
codes, counts = np.unique(fr.status, return_counts=True)
print("nodes:", mesh.n_nodes, "grid:", mesh.grid_shape)
print("status:", {STATUS_NAMES[int(c)]: int(n) for c, n in zip(codes, counts)})
print(f"median ZNCC {np.nanmedian(fr.zncc):.3f}, beta {fr.admm.beta:.3g}, ADMM steps {fr.admm.n_steps}")
finite = np.all(np.isfinite(fr.U_std), axis=1)
print("median predicted std of u, v, w [voxel]:", np.median(fr.U_std[finite], axis=0).round(4))
if truth is not None:
    from al_dvc.synthetic import evaluate_at_nodes

    U_gt = evaluate_at_nodes(truth, mesh.coordinates)
    ok = mesh.node_valid & np.all(np.isfinite(fr.U), axis=1)
    print("RMSE vs truth [voxel]:", np.sqrt(np.mean((fr.U - U_gt)[ok] ** 2, axis=0)).round(4))

## 5. Slices

`plot_field_slices` shows the three mid-planes of a node grid field. Strain
fields come from `result.result_strain[k].field(name)` (`exx`, `eyy`, `ezz`,
`exy`, `exz`, `eyz`, `e1`..`e3`, `von_mises`, `volumetric`, ...), NaN where
the plane-fit window is incomplete.

In [ ]:
strain = result.result_strain[0]
for name, values in [("u [voxel]", fr.U[:, 0]), ("exx", strain.field("exx")), ("predicted std u [voxel]", fr.U_std[:, 0])]:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    plot_field_slices(mesh.to_grid(values), mesh, title=name, fig=fig, axes=axes)

## 6. Export

`export_npz` keeps everything (`load_npz_result` reads it back), `export_vtk`
writes one `.vti` per frame plus a `.pvd` time series for ParaView,
`export_report` a PDF with parameters, convergence diagnostics and field
slices. `export_mat` writes the MATLAB ALDVC layout as well.

In [ ]:
export_npz(result, f"{OUTPUT}/results.npz")
export_vtk(result, f"{OUTPUT}/vtk", fields=["disp_magnitude", "disp_std", "exx", "eyy", "ezz", "von_mises"])
export_report(result, f"{OUTPUT}/report.pdf")

## 7. Next steps

* Masks: pass `masks=[m0, m1]` (boolean volumes, `True` = material) to
  `run_aldvc`; a mask trims the reference subsets and removes what the warped
  subsets would sample from the deformed frame.
* Sequences: `run_aldvc(para, [f0, f1, f2, ...])` with
  `reference_mode="accumulative"` or `"incremental"`; large sequences stream
  from disk through `al_dvc.io.FileVolumeProvider`.
* Command line: `al-dvc run --volumes scan/*.tif -o results --winsize 24 --step 12 --export npz vtk report`.
* Comparing with the MATLAB code: `scripts/compare_matlab.py` and
  `docs/design.md` section 10.